# House Prices - SQL Analysis Notebook

This notebook demonstrates SQL-based analysis for reporting insights.

Steps:
- Load `train.csv`
- Push data into in-memory SQLite
- Run aggregate queries and visualize results

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
SQL_FILE = PROJECT_ROOT / "sql" / "analysis_queries.sql"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PATH)

conn = sqlite3.connect(":memory:")
train.to_sql("house_prices", conn, index=False, if_exists="replace")

print("Rows loaded:", len(train))
print("Columns loaded:", train.shape[1])

## 1) Average SalePrice by Neighborhood

In [ ]:
q1 = '''
SELECT Neighborhood,
       COUNT(*) AS total_houses,
       AVG(SalePrice) AS avg_price,
       MIN(SalePrice) AS min_price,
       MAX(SalePrice) AS max_price
FROM house_prices
GROUP BY Neighborhood
HAVING COUNT(*) >= 5
ORDER BY avg_price DESC
LIMIT 15;
'''

neighborhood_df = pd.read_sql_query(q1, conn)
display(neighborhood_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=neighborhood_df.sort_values("avg_price", ascending=True),
    y="Neighborhood",
    x="avg_price",
    palette="mako",
)
plt.title("Top Neighborhood by Average SalePrice")
plt.xlabel("Average SalePrice")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sql_avg_price_by_neighborhood.png", dpi=200)
plt.show()

## 2) Average Price by OverallQual

In [ ]:
q2 = '''
SELECT OverallQual,
       COUNT(*) AS total_houses,
       AVG(SalePrice) AS avg_price
FROM house_prices
GROUP BY OverallQual
ORDER BY OverallQual;
'''

quality_df = pd.read_sql_query(q2, conn)
display(quality_df)

plt.figure(figsize=(8, 5))
sns.lineplot(data=quality_df, x="OverallQual", y="avg_price", marker="o", color="#2563eb")
plt.title("Average SalePrice by OverallQual")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sql_avg_price_by_quality.png", dpi=200)
plt.show()

## 3) Price Trend by YearBuilt

In [ ]:
q3 = '''
SELECT YearBuilt,
       COUNT(*) AS total_houses,
       AVG(SalePrice) AS avg_price
FROM house_prices
GROUP BY YearBuilt
HAVING COUNT(*) >= 2
ORDER BY YearBuilt;
'''

year_df = pd.read_sql_query(q3, conn)
display(year_df.head(20))

plt.figure(figsize=(10, 5))
sns.lineplot(data=year_df, x="YearBuilt", y="avg_price", color="#059669")
plt.title("Average SalePrice Trend by YearBuilt")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sql_avg_price_by_yearbuilt.png", dpi=200)
plt.show()

## 4) Price per Square Foot

In [ ]:
q4 = '''
SELECT Id,
       GrLivArea,
       SalePrice,
       (SalePrice * 1.0 / NULLIF(GrLivArea, 0)) AS price_per_sqft
FROM house_prices
WHERE GrLivArea IS NOT NULL
  AND SalePrice IS NOT NULL
  AND GrLivArea > 0
ORDER BY price_per_sqft DESC
LIMIT 30;
'''

sqft_df = pd.read_sql_query(q4, conn)
display(sqft_df.head(10))

plt.figure(figsize=(8, 5))
sns.boxplot(x=sqft_df["price_per_sqft"], color="#f97316")
plt.title("Price per SqFt (Top 30 rows by SQL query)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sql_price_per_sqft_box.png", dpi=200)
plt.show()

## 5) Execute Queries from sql/analysis_queries.sql

In [ ]:
if SQL_FILE.exists():
    sql_text = SQL_FILE.read_text(encoding="utf-8", errors="ignore")
    queries = [q.strip() for q in sql_text.split(";") if q.strip()]

    print("Number of parsed queries:", len(queries))
    for i, q in enumerate(queries, start=1):
        print(f"\n--- Query {i} ---")
        try:
            display(pd.read_sql_query(q, conn).head(10))
        except Exception as exc:
            print("Query execution failed:", exc)
else:
    print("sql/analysis_queries.sql not found")

## 6) Slide-ready SQL Insights

Suggested speaking points:
- Neighborhood has strong impact on average house price.
- Better overall quality is associated with higher price.
- Newer houses tend to have higher average prices.
- Very high price-per-sqft rows should be reviewed as potential outliers.

In [ ]:
conn.close()
print("SQLite connection closed.")